# Session 11 — Overfitting

**Goal:** watch a model cross the line from learning to memorising, in a single
controlled sweep — and come away with a diagnostic (the train-test gap), a way to pick
the complexity that maximises real performance, and two regularisation levers that let
you keep flexibility without paying for it.

## What this stage does for the system

Session 10 built the classifier and established the only honest way to measure it. This
session uses that measurement as an *instrument*: hold the data fixed, turn model
complexity up one notch at a time, and record both the training score and the held-out
score at every setting. The training score climbs to perfection. The held-out score
rises, peaks, and then falls. The distance between the two curves is overfitting, made
visible.

Two conclusions from that picture drive the rest of the system:

- **More flexible is not more accurate.** There is a specific complexity that maximises
  held-out performance for this registry, and going past it makes the model measurably
  worse while making it look better on every training-set metric.
- **Where the peak sits depends on how much data you have.** 297 patients support less
  complexity than 3,000 would — which makes this a *systems* constraint, not a
  hyperparameter preference.

Session 12 then asks *why* the curve has this shape, and decomposes the error into the
two competing sources that produce it.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Set up the same split as Session 10

Decision trees are the model of choice here because their complexity is controlled by a
single obvious dial — `max_depth` — where each increment doubles the number of regions
the tree can carve the patients into.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

features = [c for c in df.columns if c != "target"]
X = df[features].to_numpy()
y = df["target"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"train: {len(X_train)} patients, test: {len(X_test)} patients")
print(f"a tree of depth d can produce up to 2^d leaves:")
for d in [1, 3, 5, 10]:
    print(f"  depth {d:2}: up to {2 ** d:5} leaves for {len(X_train)} training patients")

**Observe:** at depth 10 a tree could carve out 1,024 leaves for 237 training patients —
four times as many regions as there are patients to fill them.
**Infer:** that arithmetic is overfitting waiting to happen. Once the number of
available regions exceeds the number of patients, the tree can place each patient in a
region of their own and achieve perfect training accuracy without discovering anything
that transfers. The capacity crossover happens around depth 8 here, which predicts
roughly where the curve in Step 3 will flatten at 1.0. Note this is the same
capacity-versus-data argument from Session 10's Step 3, now with a dial attached so it
can be swept rather than asserted.

## Step 3 — Sweep complexity and watch both curves

One tree per depth, trained identically, scored on both the training and the held-out
patients.

In [ ]:
import matplotlib.pyplot as plt

depths = range(1, 16)
rows = []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    rows.append({
        "depth": d,
        "leaves": tree.get_n_leaves(),
        "train_AUC": roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1]),
        "test_AUC": roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]),
    })
results = pd.DataFrame(rows)
results["gap"] = results["train_AUC"] - results["test_AUC"]
print(results.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results["depth"], results["train_AUC"], "o-", color="steelblue", label="training AUC")
ax.plot(results["depth"], results["test_AUC"], "o-", color="indianred", label="held-out AUC")
best = results.loc[results["test_AUC"].idxmax()]
ax.axvline(best["depth"], color="green", ls="--", label=f"best held-out (depth {int(best['depth'])})")
ax.set_xlabel("max_depth (model complexity)"); ax.set_ylabel("AUC")
ax.set_title("Training score always improves. Held-out score does not.")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** training AUC climbs `0.763 → 1.000` and stays there from depth 7 onward,
while held-out AUC peaks at `0.852` at **depth 2** and then declines to `0.732` — worse
than the depth-1 stump.
**Infer:** everything past depth 2 is the model learning things that are true of these
237 patients and false of patients in general. Two details are worth extracting. The
peak is *early* — a two-level tree, four leaves, on thirteen inputs — which is what 237
patients supports and a direct illustration of complexity being bounded by data volume
rather than by the number of available inputs. And the decline is not small: from
`0.852` down to `0.732` is giving back most of what the model learned in the first
place. A practitioner who selected depth by training score would land on depth 7+ and
ship the worst model in the table while holding a perfect `1.000` to justify it.

## Step 4 — The train-test gap as an alarm

The gap between the two curves is the direct measure of overfitting, and it is
monitorable in production in a way that "is my model overfitting?" is not.

In [ ]:
print(results[["depth", "leaves", "train_AUC", "test_AUC", "gap"]].round(3).to_string(index=False))

best_row = results.loc[results["test_AUC"].idxmax()]
worst_row = results.loc[results["test_AUC"].idxmin()]
print(f"\nbest held-out:  depth {int(best_row['depth'])}, "
      f"{int(best_row['leaves'])} leaves, test AUC {best_row['test_AUC']:.3f}, gap {best_row['gap']:+.3f}")
print(f"worst held-out: depth {int(worst_row['depth'])}, "
      f"{int(worst_row['leaves'])} leaves, test AUC {worst_row['test_AUC']:.3f}, gap {worst_row['gap']:+.3f}")
print(f"\npatients per leaf at the best depth:  {len(X_train) / best_row['leaves']:.1f}")
print(f"patients per leaf at the worst depth: {len(X_train) / worst_row['leaves']:.1f}")

**Observe:** the gap widens from `+0.004` at depth 1 to `+0.268` at the deepest
settings (with one non-monotone step at depth 7), and patients-per-leaf collapses from
`59.2` at the best depth to `5.5` at the worst.
**Infer:** patients-per-leaf is the more actionable of the two numbers, because it is
computable without a test set at all: a leaf holding six patients is estimating a
disease probability from six observations, which Session 4's standard-error argument
says is nearly worthless. The gap itself makes a good deployment alarm — a model whose
training and held-out scores diverge is a model whose reported performance will not
survive contact with new patients. But note the gap is a *symptom*, not the objective:
depth 1 has the smallest gap of all and is not the best model, because it underfits.
Maximise held-out performance; use the gap to understand why the maximum is where it is.

## Step 5 — Seeing it directly: decision boundaries

Two inputs instead of thirteen, so the model's carve-up of the space can be plotted.
The pathology is visible without any metric at all.

In [ ]:
from matplotlib.colors import ListedColormap

pair = ["age", "thalach"]
X2 = df[pair].to_numpy()
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y)

xx, yy = np.meshgrid(
    np.linspace(X2[:, 0].min() - 2, X2[:, 0].max() + 2, 300),
    np.linspace(X2[:, 1].min() - 5, X2[:, 1].max() + 5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, depth in zip(axes, [1, 3, None]):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X2_train, y2_train)
    zz = tree.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=0.6)
    ax.scatter(X2_train[:, 0], X2_train[:, 1], c=y2_train,
               cmap=ListedColormap(["steelblue", "darkred"]), s=20, edgecolor="white", linewidth=0.4)
    tr = roc_auc_score(y2_train, tree.predict_proba(X2_train)[:, 1])
    te = roc_auc_score(y2_test, tree.predict_proba(X2_test)[:, 1])
    ax.set_xlabel(pair[0]); ax.set_ylabel(pair[1])
    ax.set_title(f"depth={depth}, leaves={tree.get_n_leaves()}\ntrain {tr:.3f} / test {te:.3f}")
plt.tight_layout()
plt.show()

**Observe:** depth 1 splits the plane with a single straight line; depth 3 produces a
handful of rectangular blocks that follow the broad trend; the unrestricted tree
fragments into dozens of thin slivers, several of them wrapping individual points.
**Infer:** each sliver is a claim that patients in that narrow band of age and heart rate
have a specific disease probability — a claim supported by the one or two patients who
happen to sit there. That is memorisation with a picture attached, and it is why the
unrestricted panel's train/test numbers diverge so violently. This view also exposes a
structural property of trees: every boundary is axis-parallel, so a genuinely diagonal
relationship has to be approximated by a staircase, and *some* of the fragmentation is
the model straining against that limitation rather than pure overfitting. Logistic
regression, whose boundary is a single diagonal line, has the opposite bias — which is
Session 12's subject.

## Step 6 — Regularisation: keep the flexibility, drop the memorisation

`max_depth` is a blunt instrument — it caps every branch equally, whether that branch
has 100 patients or 3. `min_samples_leaf` constrains the model where the problem
actually is, refusing to create any leaf backed by too few patients.

In [ ]:
variants = [
    ("unrestricted",           dict()),
    ("max_depth=2",            dict(max_depth=2)),
    ("min_samples_leaf=20",    dict(min_samples_leaf=20)),
    ("min_samples_leaf=20, depth=5", dict(min_samples_leaf=20, max_depth=5)),
    ("ccp_alpha=0.02 (pruning)",     dict(ccp_alpha=0.02)),
]

rows = []
for name, params in variants:
    tree = DecisionTreeClassifier(random_state=0, **params).fit(X_train, y_train)
    tr = roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1])
    te = roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1])
    rows.append({"model": name, "leaves": tree.get_n_leaves(),
                 "train_AUC": tr, "test_AUC": te, "gap": tr - te})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Observe:** the unrestricted tree has 43 leaves with a `0.268` gap and test AUC
`0.732`; `min_samples_leaf=20` cuts it to 9 leaves, a gap near `−0.010`, and test AUC
`0.894` — better than the best depth-limited tree in Step 3.
**Infer:** the leaf-size constraint beats the depth constraint because it targets the
actual failure. Depth limiting is uniform: it refuses a fourth split even where 80
patients would support one, while still permitting a third split on 4 patients. Leaf
size lets the tree stay deep where the data is dense and stops it where the data runs
out, which is why 9 well-populated leaves outperform 43 sparse ones. `ccp_alpha`
(cost-complexity pruning) attacks the same problem from the other end — grow the tree
fully, then prune back the branches that do not pay for themselves — and is generally the
more principled option, since it decides what to remove by measured contribution rather
than by a structural rule chosen in advance.

## Step 7 — Choosing complexity properly: cross-validated selection

Steps 3-6 picked hyperparameters by looking at test-set scores — which is Session 10's
Step 7 leakage-through-model-selection, committed in full view. The correct procedure
selects on cross-validation *within the training data*, leaving the test set untouched
for a single final measurement.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_grid={"max_depth": [1, 2, 3, 4, 5, 8, None],
                "min_samples_leaf": [1, 5, 10, 20, 30]},
    scoring="roc_auc",
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
).fit(X_train, y_train)

print(f"selected by CV on training data only: {grid.best_params_}")
print(f"cross-validated AUC during selection: {grid.best_score_:.3f}")

final_auc = roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1])
print(f"\nfinal AUC on the untouched test set:  {final_auc:.3f}")
print(f"best test AUC found by peeking in Step 3: {results['test_AUC'].max():.3f}")

**Observe:** cross-validation selects `max_depth=3, min_samples_leaf=10`, scoring
`0.829` during selection, and the untouched test set then returns `0.871` — above both
the selection estimate and the `0.852` that Steps 3-6 found by peeking.
**Infer:** this is the procedure to actually use, and its virtue is that the final number
is trustworthy, not that it is high — here it happened to come out high, and the reverse
happens just as often. Two separate things are visible. Steps 3-6 found their peak by
trying 15 depths against the test set and keeping the winner, so `0.852` is a maximum
over 15 draws and biased upward as an estimate of that model's true performance — the
same mechanism as Session 6's multiple-comparisons problem, applied to hyperparameters
instead of hypotheses. And the `0.829` versus `0.871` spread is Session 10's Step 4
again: a 60-patient test set has a standard error large enough to swamp differences of
this size, in either direction. Neither observation changes the rule: **select on
cross-validation, report on data touched exactly once**, and quote the fold spread so
the reader knows how much of the number is noise.

## What this session hands to the next one

- **A complexity setting chosen honestly**, and the curve showing what happens on either
  side of it.
- **The train-test gap** as a monitorable overfitting alarm, with the caveat that a
  small gap is not the objective.
- **Regularisation** — `min_samples_leaf`, `ccp_alpha` — as the way to keep flexibility
  where the data supports it.
- **An open question**: the held-out curve rises and *then* falls, so two opposing forces
  must be at work. Session 12 names them (bias and variance), measures each separately
  on this registry, and shows which one to attack.

## Try it yourself

1. Re-run Step 3 with `random_state=7` on the split. Does the peak stay at depth 2? What
   does the answer say about selecting complexity from a single split?
2. Train on 60 patients instead of 237 and repeat the sweep. Which direction does the
   optimal depth move, and does that match the capacity argument in Step 2?
3. Swap the tree for `LogisticRegression` and sweep `C` from 0.001 to 1000 (smaller C =
   stronger regularisation). Does the same rise-then-fall shape appear?
4. In Step 5, replace `thalach` with `oldpeak` and re-plot. Does the unrestricted tree
   fragment more or less, and how does the zero-inflation from Session 3 show up in the
   boundaries?